# 2단계 ⓐ — HI-Small 피처 계산 (20_small_features)
2단계(모델 비교) 는 **HI-Small** 에서 돈다. 1단계에서 확정한 피처 셋(07 `tod`: 기본 3 + 시간대 3 + pair 4 + 활동 8 + 이력 12 = 30, GNN 은 + port 2) 을
Small 에 **한 번** 계산해 두고, 21(GNN 6종 × 시드 3) 과 22(트리·스태킹) 가 같은 행렬을 읽는다.

- SQL 은 02·03·06·07 노트북의 것을 그대로 (누수 규칙 동일: 거래 시각 기준 과거만, 같은 분 제외, 누적은 전날까지 ASOF)
- Small 은 500만 건이라 메모리 걱정 없음 — 그래도 파이프라인은 1단계와 같은 함수(`load_tx_lowmem`, `znorm_features(out_path)`) 를 쓴다
- 분할: 일 단위 60/20/20 (`bc_lib.WINDOWS['HI-Small']`, Multi-GNN 규칙). 검증 표본은 양성 전부 + 정상 **5%** (Large 는 2% — Small 은 양성이 적어 선택이 흔들리지 않게 조금 올림)
- 결과: `parquets/HI-Small/features_bc/` 에 피처 parquet 4개 + `feat16_stage2.npy` + 표본 parquet + `stage2_meta.json`

## 0. 작업 준비

In [ ]:
import sys, json, time, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd()))
import bc_lib as B
L = B.L

In [ ]:
RUN_NAME = 'stage2'
cfg = B.Config(dataset='HI-Small', smoke=False, val_neg_rate=0.05)
B.seed_all(cfg.seed)
# ---- 2단계 피처 셋 = 07 `tod` (timestamp → 시간대, amount_seen 제외) --------------------------------------
PAIR_FEATURES = ['pair_prior_count', 'pair_gap_minutes', 'pair_count_1h', 'reverse_history_count']                                   # 02
ACT_FEATURES  = ['src_out_1h', 'src_out_1d', 'src_uniq_dst_1d', 'src_out_amt_1d', 'dst_in_1h', 'dst_in_1d', 'dst_uniq_src_1d', 'dst_in_amt_1d']   # 03
HIST_FEATURES = ['src_hist_out', 'src_hist_uniq_out', 'src_active_days', 'src_rate_vs_prior', 'src_minute_burst', 'src_gap_minutes',          # 06
                 'dst_hist_in', 'dst_hist_uniq_in', 'dst_active_days', 'dst_rate_vs_prior', 'dst_minute_burst', 'dst_gap_minutes']
TOD_FEATURES  = ['hour_sin', 'hour_cos', 'weekday']                                                                                    # 07
BASIC         = ['amount_received', 'received_currency', 'payment_format']                                                            # 기본 4 − timestamp
TREE_FEATURES = BASIC + TOD_FEATURES + PAIR_FEATURES + ACT_FEATURES + HIST_FEATURES                        # 30
GNN_FEATURES  = BASIC + TOD_FEATURES + B.PORT_FEATURES + PAIR_FEATURES + ACT_FEATURES + HIST_FEATURES      # 32 = 피처 행렬(FEAT16) 열 순서. port 는 6, 7 번
PORT_COLS     = [GNN_FEATURES.index(c) for c in B.PORT_FEATURES]
TREE_COLS     = [GNN_FEATURES.index(c) for c in TREE_FEATURES]
PATHS = {'pair': cfg.feat_dir / 'pair_history.parquet', 'act': cfg.feat_dir / 'account_activity.parquet',
         'hist': cfg.feat_dir / 'account_history.parquet', 'tod': cfg.feat_dir / 'time_of_day.parquet'}
FEAT_PATH = cfg.feat_dir / f'feat16_{RUN_NAME}.npy'          # z-norm 행렬 (float16, GNN_FEATURES 순)
META_PATH = cfg.feat_dir / f'{RUN_NAME}_meta.json'            # 20 이 쓰고 21·22 가 읽는 설정 (피처 순서·표본 파일)
SAMPLE_PATHS = {k: cfg.feat_dir / f'{RUN_NAME}_{k}.parquet' for k in ('val', 'test', 'indegree')}
print('분할:', cfg.train_window, '|', cfg.val_window, '|', cfg.test_window)
len(TREE_FEATURES), len(GNN_FEATURES), PORT_COLS

## 1. 데이터 확인 — 날짜별 건수·양성, 패턴 분포 (Small 이 10일짜리인지, 분할이 맞는지)

In [ ]:
con = L.connect(threads=8, memory='8GB')
daily = con.execute(f"""
    SELECT day, count(*) AS n, sum(is_laundering::INT) AS n_pos, round(100.0 * avg(is_laundering::INT), 3) AS pos_pct
    FROM read_parquet({L.q(cfg.tx_path)}) GROUP BY 1 ORDER BY 1""").df()
a, b = L.parse_window(cfg.train_window); c, d = L.parse_window(cfg.val_window); e, f = L.parse_window(cfg.test_window)
daily['split'] = np.select([daily.day.astype(str).between(a, b), daily.day.astype(str).between(c, d), daily.day.astype(str).between(e, f)], ['train', 'val', 'test'], '-')
display(daily)
daily.groupby('split')[['n', 'n_pos']].sum()

In [ ]:
# 패턴 유형별 양성 수 (patterns.parquet) — 1단계 Large 와 비율 비교용
pat = con.execute(f"SELECT coalesce(pattern_type, 'UNMATCHED') AS pattern_type, count(DISTINCT edge_id) AS n FROM read_parquet({L.q(L.BASE / 'patterns.parquet')}) GROUP BY 1 ORDER BY 2 DESC").df()
n_pos = con.execute(f"SELECT sum(is_laundering::INT) FROM read_parquet({L.q(cfg.tx_path)})").fetchone()[0]
pat['share'] = (pat.n / n_pos).round(3); print('양성 전체', n_pos, ' 패턴 파일 안 양성', pat.n.sum(), ' → UNMATCHED 비율', round(1 - pat.n.sum() / n_pos, 3)); pat

## 2. 코드표 + port numbering (Multi-GNN ports)

In [ ]:
currency_codes, format_codes = B.code_tables(con, cfg)
PORTS_PATH = B.ensure_ports(con, cfg)          # features_ports/ports.parquet — 있으면 건너뜀
print(currency_codes.shape, format_codes.shape, PORTS_PATH)

## 3. 02 pair history — 같은 쌍 반복 (4)

In [ ]:
if not PATHS['pair'].exists():
    t0 = time.time()
    con.execute(f"""
        CREATE OR REPLACE TABLE tx_min AS
        SELECT edge_id, src, dst, ts, day FROM read_parquet({L.q(cfg.tx_path)})""")
    con.execute("""
        CREATE OR REPLACE TABLE pair_win AS
        SELECT edge_id, ts,
               count(*) OVER (PARTITION BY src, dst ORDER BY ts RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL '1 minute' PRECEDING) AS pair_prior,
               max(ts)  OVER (PARTITION BY src, dst ORDER BY ts RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL '1 minute' PRECEDING) AS pair_last,
               count(*) OVER (PARTITION BY src, dst ORDER BY ts RANGE BETWEEN INTERVAL '60 minutes' PRECEDING AND INTERVAL '1 minute' PRECEDING) AS pair_hour
        FROM tx_min""")
    con.execute("""
        CREATE OR REPLACE TABLE pair_state AS
        SELECT src, dst, day, sum(n) OVER (PARTITION BY src, dst ORDER BY day ROWS UNBOUNDED PRECEDING) AS history_count
        FROM (SELECT src, dst, day, count(*) n FROM tx_min GROUP BY src, dst, day)""")
    con.execute("""
        CREATE OR REPLACE TABLE pairs AS
        SELECT t.src, t.dst, t.day, coalesce(r.history_count, 0) AS reverse_n
        FROM (SELECT DISTINCT src, dst, day FROM tx_min) t
        ASOF LEFT JOIN pair_state r ON t.src = r.dst AND t.dst = r.src AND t.day > r.day""")
    con.execute(f"""
        COPY (
            SELECT t.edge_id,
                   ln(1 + w.pair_prior)::FLOAT                                                                           AS pair_prior_count,
                   (CASE WHEN w.pair_last IS NULL THEN 0 ELSE ln(2 + date_diff('minute', w.pair_last, t.ts)) END)::FLOAT AS pair_gap_minutes,
                   ln(1 + w.pair_hour)::FLOAT                                                                            AS pair_count_1h,
                   ln(1 + p.reverse_n)::FLOAT                                                                            AS reverse_history_count
            FROM tx_min t JOIN pair_win w USING (edge_id)
            JOIN pairs p ON t.src = p.src AND t.dst = p.dst AND t.day = p.day
            ORDER BY t.edge_id
        ) TO {L.q(PATHS['pair'])} (FORMAT PARQUET, COMPRESSION ZSTD)""")
    con.execute("DROP TABLE pair_win; DROP TABLE pair_state; DROP TABLE pairs; DROP TABLE tx_min")
    print(f"{PATHS['pair'].name} 생성 {time.time()-t0:.0f}s")
con.execute(f"SELECT count(*), {', '.join(f'avg({c}) AS avg_{c}' for c in PAIR_FEATURES)} FROM read_parquet({L.q(PATHS['pair'])})").df().T

## 4. 03 account activity — 계좌 단기 활동 (8)

In [ ]:
if not PATHS['act'].exists():
    t0 = time.time()
    con.execute(f"""
        CREATE OR REPLACE TABLE tx_min AS
        SELECT edge_id, src, dst, ts, day, received FROM read_parquet({L.q(cfg.tx_path)})""")
    for acct, other, pre in (('src', 'dst', 'out'), ('dst', 'src', 'in')):
        con.execute(f"""
            CREATE OR REPLACE TABLE {acct}_win AS
            SELECT edge_id,
                   count(*)            OVER w1h AS {pre}_1h,
                   count(*)            OVER w1d AS {pre}_1d,
                   sum(first_pair_day) OVER w1d AS uniq_1d,
                   sum(received)       OVER w1d AS amt_1d
            FROM (SELECT *, CASE WHEN row_number() OVER (PARTITION BY {acct}, {other}, day ORDER BY ts, edge_id) = 1 THEN 1 ELSE 0 END AS first_pair_day FROM tx_min)
            WINDOW w1h AS (PARTITION BY {acct} ORDER BY ts RANGE BETWEEN INTERVAL '60 minutes' PRECEDING AND INTERVAL '1 minute' PRECEDING),
                   w1d AS (PARTITION BY {acct} ORDER BY ts RANGE BETWEEN INTERVAL '24 hours'  PRECEDING AND INTERVAL '1 minute' PRECEDING)""")
        print(f'  {acct}_win {time.time()-t0:.0f}s')
    con.execute(f"""
        COPY (
            SELECT t.edge_id,
                   ln(1 + s.out_1h)::FLOAT                AS src_out_1h,
                   ln(1 + s.out_1d)::FLOAT                AS src_out_1d,
                   ln(1 + coalesce(s.uniq_1d, 0))::FLOAT  AS src_uniq_dst_1d,
                   ln(1 + coalesce(s.amt_1d, 0))::FLOAT   AS src_out_amt_1d,
                   ln(1 + d.in_1h)::FLOAT                 AS dst_in_1h,
                   ln(1 + d.in_1d)::FLOAT                 AS dst_in_1d,
                   ln(1 + coalesce(d.uniq_1d, 0))::FLOAT  AS dst_uniq_src_1d,
                   ln(1 + coalesce(d.amt_1d, 0))::FLOAT   AS dst_in_amt_1d
            FROM tx_min t JOIN src_win s USING (edge_id) JOIN dst_win d USING (edge_id)
            ORDER BY t.edge_id
        ) TO {L.q(PATHS['act'])} (FORMAT PARQUET, COMPRESSION ZSTD)""")
    con.execute("DROP TABLE src_win; DROP TABLE dst_win; DROP TABLE tx_min")
    print(f"{PATHS['act'].name} 생성 {time.time()-t0:.0f}s")
con.execute(f"SELECT count(*), {', '.join(f'avg({c}) AS avg_{c}' for c in ACT_FEATURES)} FROM read_parquet({L.q(PATHS['act'])})").df().T

## 5. 06 account history — 계좌 이력·평소 대비 (12; amount_seen 은 계산만, 안 씀)

In [ ]:
if not PATHS['hist'].exists():
    t0 = time.time()
    con.execute(f"""
        CREATE OR REPLACE TABLE tx_min AS
        SELECT edge_id, src, dst, ts, day, received FROM read_parquet({L.q(cfg.tx_path)})""")
    # (a) 일 단위 계좌 상태 (송신 역할 / 수신 역할) → (계좌, 날짜) 마다 전날까지 상태를 ASOF 로
    for role, acct_col, other_col in [('out', 'src', 'dst'), ('in', 'dst', 'src')]:
        con.execute(f"""
            CREATE OR REPLACE TABLE {role}_state AS
            WITH d AS (SELECT {acct_col} AS acct, day, count(*) AS n FROM tx_min GROUP BY 1, 2),
                 p AS (SELECT {acct_col} AS acct, day, count(*) AS new_partners
                       FROM (SELECT {acct_col}, {other_col}, min(day) AS day FROM tx_min GROUP BY 1, 2) GROUP BY 1, 2)
            SELECT d.acct, d.day, sum(d.n) OVER w AS cum_cnt, sum(coalesce(p.new_partners, 0)) OVER w AS cum_uniq, count(*) OVER w AS active_days
            FROM d LEFT JOIN p USING (acct, day)
            WINDOW w AS (PARTITION BY d.acct ORDER BY d.day ROWS UNBOUNDED PRECEDING)""")
        con.execute(f"""
            CREATE OR REPLACE TABLE {acct_col}_hist AS
            SELECT t.{acct_col} AS acct, t.day, s.cum_cnt, s.cum_uniq, s.active_days
            FROM (SELECT DISTINCT {acct_col}, day FROM tx_min) t
            ASOF LEFT JOIN {role}_state s ON t.{acct_col} = s.acct AND t.day > s.day""")
        con.execute(f"DROP TABLE {role}_state")
    # (b) 창 함수 — 같은 분 건수, 직전 60분·24h 건수, 직전 거래 시각, 같은 금액의 첫 거래일
    for acct_col in ['src', 'dst']:
        con.execute(f"""
            CREATE OR REPLACE TABLE {acct_col}_win AS
            SELECT edge_id,
                   count(*) OVER (PARTITION BY {acct_col}, ts) - 1          AS same_min,
                   count(*) OVER w1h                                        AS n_1h,
                   count(*) OVER w1d                                        AS n_1d,
                   max(ts)  OVER wall                                       AS last_ts,
                   min(day) OVER (PARTITION BY {acct_col}, received)        AS amt_first_day
            FROM tx_min
            WINDOW w1h  AS (PARTITION BY {acct_col} ORDER BY ts RANGE BETWEEN INTERVAL '60 minutes' PRECEDING AND INTERVAL '1 minute' PRECEDING),
                   w1d  AS (PARTITION BY {acct_col} ORDER BY ts RANGE BETWEEN INTERVAL '24 hours'  PRECEDING AND INTERVAL '1 minute' PRECEDING),
                   wall AS (PARTITION BY {acct_col} ORDER BY ts RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL '1 minute' PRECEDING)""")
    # (c) ln 변환·조합 후 저장 (edge_id 순)
    def cols(a, hist_name, uniq_name):
        return f"""
                   ln(1 + coalesce({a}h.cum_cnt, 0))::FLOAT                                                                 AS {a}_{hist_name},
                   ln(1 + coalesce({a}h.cum_uniq, 0))::FLOAT                                                                AS {a}_{uniq_name},
                   ln(1 + coalesce({a}h.active_days, 0))::FLOAT                                                             AS {a}_active_days,
                   (CASE WHEN {a}h.active_days IS NULL THEN 0 ELSE ln(1 + {a}w.n_1d) - ln(1 + {a}h.cum_cnt / {a}h.active_days) END)::FLOAT AS {a}_rate_vs_prior,
                   (ln(1 + 60 * {a}w.same_min) - ln(1 + {a}w.n_1h))::FLOAT                                                   AS {a}_minute_burst,
                   (CASE WHEN {a}w.last_ts IS NULL THEN 0 ELSE ln(2 + date_diff('minute', {a}w.last_ts, t.ts)) END)::FLOAT    AS {a}_gap_minutes,
                   (CASE WHEN {a}w.amt_first_day < t.day THEN 1 ELSE 0 END)::FLOAT                                          AS {a}_amount_seen"""
    con.execute(f"""
        COPY (
            SELECT t.edge_id, {cols('src', 'hist_out', 'hist_uniq_out')}, {cols('dst', 'hist_in', 'hist_uniq_in')}
            FROM tx_min t
            JOIN src_win srcw USING (edge_id) JOIN dst_win dstw USING (edge_id)
            JOIN src_hist srch ON t.src = srch.acct AND t.day = srch.day
            JOIN dst_hist dsth ON t.dst = dsth.acct AND t.day = dsth.day
            ORDER BY t.edge_id
        ) TO {L.q(PATHS['hist'])} (FORMAT PARQUET, COMPRESSION ZSTD)""")
    con.execute("DROP TABLE src_win; DROP TABLE dst_win; DROP TABLE src_hist; DROP TABLE dst_hist; DROP TABLE tx_min")
    print(f"{PATHS['hist'].name} 생성 {time.time()-t0:.0f}s")
con.execute(f"SELECT count(*), {', '.join(f'avg({c}) AS avg_{c}' for c in HIST_FEATURES)} FROM read_parquet({L.q(PATHS['hist'])})").df().T

## 6. 07 time of day — 시간대 (3)

In [ ]:
if not PATHS['tod'].exists():
    t0 = time.time()
    con.execute(f"""
        COPY (
            SELECT edge_id,
                   sin(2 * pi() * (hour(ts) * 60 + minute(ts)) / 1440.0)::FLOAT AS hour_sin,
                   cos(2 * pi() * (hour(ts) * 60 + minute(ts)) / 1440.0)::FLOAT AS hour_cos,
                   ((dayofweek(ts) + 6) % 7)::FLOAT                            AS weekday      -- 월요일 = 0
            FROM read_parquet({L.q(cfg.tx_path)}) ORDER BY edge_id
        ) TO {L.q(PATHS['tod'])} (FORMAT PARQUET, COMPRESSION ZSTD)""")
    print(f"{PATHS['tod'].name} 생성 {time.time()-t0:.0f}s")
con.execute(f"SELECT weekday, count(*) n, avg(hour_sin) s, avg(hour_cos) c FROM read_parquet({L.q(PATHS['tod'])}) GROUP BY 1 ORDER BY 1").df()

## 7. 피처가 의도대로 나왔나 — 라벨별 평균 (Large 06 표와 방향이 같아야 함)

In [ ]:
q = ', '.join(f'avg(f.{c}) AS {c}' for c in PAIR_FEATURES + ACT_FEATURES + HIST_FEATURES)
label_mean = con.execute(f"""
    WITH f AS (SELECT * FROM read_parquet({L.q(PATHS['pair'])}) pair JOIN read_parquet({L.q(PATHS['act'])}) act USING (edge_id)
                                                                   JOIN read_parquet({L.q(PATHS['hist'])}) hist USING (edge_id))
    SELECT t.is_laundering::INT AS y, count(*) AS n, {q}
    FROM read_parquet({L.q(cfg.tx_path)}) t JOIN f USING (edge_id) GROUP BY 1 ORDER BY 1""").df().set_index('y').T
label_mean.columns = ['normal', 'laundering']; label_mean['diff'] = label_mean.laundering - label_mean.normal
label_mean.round(3)

## 8. z-norm 행렬 (학습 구간 통계) — `feat16_stage2.npy`, GNN_FEATURES 열 순서

In [ ]:
tx = B.load_tx_lowmem(con, cfg, PORTS_PATH)                 # features_bc/tx_base.parquet 생성 후 열 단위 적재 (edge_id 순)
tx = B.attach_parquet(tx, PATHS['tod'],  TOD_FEATURES,  dtype=np.float16)
tx = B.attach_parquet(tx, PATHS['pair'], PAIR_FEATURES, dtype=np.float16)
tx = B.attach_parquet(tx, PATHS['act'],  ACT_FEATURES,  dtype=np.float16)
tx = B.attach_parquet(tx, PATHS['hist'], HIST_FEATURES, dtype=np.float16)
tx = tx.drop(columns=['timestamp'])                                # 07 결정: 절대 시각은 안 쓴다
B.trim('(피처 붙인 후)'); tx.shape

In [ ]:
in_train = B.window_mask(tx, cfg.train_window)
feat_z, znorm_table = B.znorm_features(tx, GNN_FEATURES, in_train, out_path=FEAT_PATH)
znorm_table.to_csv(cfg.feat_dir / f'{RUN_NAME}_znorm.csv', index=False, encoding='utf-8-sig')
print(feat_z.shape, feat_z.dtype, FEAT_PATH); znorm_table

## 9. 검증·테스트 표본, in-degree — 저장 (21·22 가 그대로 읽는다)

In [ ]:
val_df  = B.val_sample(con, cfg.val_window, cfg.val_neg_rate)
test_df = B.test_full(con, cfg.test_window)
in_degree = B.recipient_indegree(con, cfg)
val_df.to_parquet(SAMPLE_PATHS['val'], index=False); test_df.to_parquet(SAMPLE_PATHS['test'], index=False)
in_degree.rename_axis('node_id').reset_index().to_parquet(SAMPLE_PATHS['indegree'], index=False)
all_edge_ids = tx['edge_id'].to_numpy()
val_idx = B.rows_of(all_edge_ids, val_df['edge_id'].to_numpy()); test_idx = B.rows_of(all_edge_ids, test_df['edge_id'].to_numpy())
assert (tx['y'].to_numpy()[val_idx] == val_df['y'].to_numpy()).all() and (tx['y'].to_numpy()[test_idx] == test_df['y'].to_numpy()).all()
print(f"학습 target {in_train.sum():,} (양성 {int(tx['y'].to_numpy()[in_train].sum()):,})  검증 표본 {len(val_df):,} (양성 {int(val_df.y.sum()):,})  테스트 전체 {len(test_df):,} (양성 {int(test_df.y.sum()):,})")
META_PATH.write_text(json.dumps(dict(run=RUN_NAME, dataset=cfg.dataset, val_neg_rate=cfg.val_neg_rate, gnn_features=GNN_FEATURES, tree_features=TREE_FEATURES,
                                     port_cols=PORT_COLS, tree_cols=TREE_COLS, feat_path=str(FEAT_PATH), ports_path=str(PORTS_PATH),
                                     samples={k: str(v) for k, v in SAMPLE_PATHS.items()}, windows=dict(train=cfg.train_window, val=cfg.val_window, test=cfg.test_window),
                                     n_rows=len(tx)), ensure_ascii=False, indent=2), encoding='utf-8')
con.close(); print('meta →', META_PATH)

## 10. 다음
- `21_model_compare.ipynb`: GNN 6종 × 시드 3 (GINe · GINe-128 · PNA · GAT · TransConv · SALT-Trans), 같은 프로토콜
- `22_stacking_summary.ipynb`: 트리(XGB · LGB) × 시드 3, GNN + 트리 스태킹, 종합표 → 상위 2개 선정 (3단계 Large)